In [2]:
# Librerias
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
import torch
import requests
from bs4 import BeautifulSoup

/home/alejo/proyectos/Agente-Consulta-Paginas-Web/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-17 15:26:50.766341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747513610.855588   14134 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747513610.880431   14134 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747513611.076019   14134 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the 

In [3]:
# Cargar modelos
generator = pipeline("text2text-generation", model="google/flan-t5-base") # o "google/flan-t5-small"
llm = HuggingFacePipeline(pipeline=generator)
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

Device set to use cpu
/tmp/ipykernel_14134/2371647568.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


In [4]:
urls = [
    "https://seaborn.pydata.org/",
    "https://matplotlib.org/",
    "https://plotly.com/python/getting-started/"
]

# 🔍 Función para obtener el texto de cada web
def get_page_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    texts = soup.stripped_strings
    return ' '.join(texts)

# Obtenemos los textos
texts = [get_page_text(url) for url in urls]

In [5]:
texts

['seaborn: statistical data visualization — seaborn 0.13.2 documentation Ctrl + K Installing Gallery Tutorial API Releases Citing FAQ GitHub StackOverflow Twitter Site Navigation Installing Gallery Tutorial API Releases Citing FAQ GitHub StackOverflow Twitter seaborn: statistical data visualization # Seaborn is a Python data visualization library based on matplotlib . It provides a high-level interface for drawing\nattractive and informative statistical graphics. For a brief introduction to the ideas behind the library, you can read the introductory notes or the paper . Visit the installation page to see how you can download the package\nand get started with it. You can browse the example gallery to see some of the things that you can do with seaborn,\nand then check out the tutorials or API reference to find out how. To see the code or report a bug, please visit the GitHub repository . General support questions are most at home\non stackoverflow , which\nhas a dedicated channel for se

In [6]:
# Crear embeddings de contexto o sea el RAG
vector_db = FAISS.from_texts(texts=texts, embedding=embedding_model)
retriever = vector_db.as_retriever()

In [16]:
class AutonomousAgentFSM:
    def __init__(self, goal, llm, retriever):
        self.goal = goal
        self.llm = llm
        self.retriever = retriever
        self.qa_chain = RetrievalQA.from_chain_type(llm=self.llm, retriever=self.retriever, chain_type="stuff")
        self.state = "planificar"
        self.subtasks = []
        self.results = []
        self.evaluation = ""
        #self.texts = texts

    def plan(self):
        # Estas son las subtareas que el agente va a realizar
        # En este caso, el agente va a buscar información sobre las tres librerías
        # y va a comparar las tres librerías
        # y va a recomendar la mejor para uso general
        # y va a explicar su elección
        return [
            f"Search information about {self.goal}", #{self.texts} 
            "Compare the data of three visualization libraries", # 1. Seaborn: {self.texts[0]}, 2. Matplotlib: {self.texts[1]}, 3. Plotly:{self.texts[2]}
            #"Create a markdown table with the comparison",
            "recommend the best visualization library for python"
            #,"Explain your choice"
        ]
    
    # # Esta función es la que se encarga de dividir el objetivo en subtareas
    # # Dependiendo del objetivo, el agente puede dividirlo en subtareas
    # # más complejas o más simples
    #def plan(self):
    #    prompt = f"Divide the following objective into clearly and orderly subtasks: {self.goal}"
    #    plan_output = self.llm(prompt)
    #    subtasks = [line.strip("-• ") for line in plan_output.split("\n") if line.strip()]
    #    return subtasks


    def execute(self, task):
        print(f"🔧 Ejecutando subtarea: {task}")
        return self.qa_chain.run(task)

    def evaluate(self, results):
        return f"✅ Evaluación de resultados:\n- " + "\n- ".join(results)

    def run(self):
        while self.state != "final":
            if self.state == "planificar":
                self.subtasks = self.plan()
                self.state = "ejecutar"
            elif self.state == "ejecutar":
                for task in self.subtasks:
                    result = self.execute(task)
                    self.results.append(result)
                self.state = "evaluar"
            elif self.state == "evaluar":
                self.evaluation = self.evaluate(self.results)
                self.state = "final"
        return self.evaluation

In [17]:
prompt = (
"Compare the three visualization libraries considering:\n"
"- Ease of use\n- Aesthetics\n- Personalization\n"
"- Interactivity\n- Statistical support\n- Jupyter/Dash integration\n\n"
"Make a final one recommendation for general use."
)

#prompt = "Compare the three visualization libraries and choose the best one for general use."

#prompt = "Compare the three visualization libraries and show if are easy of use, aesthetics, personalization, interactivity, statistical support, jupyter or dash integration and last choose the best one for general use."

#prompt = (
    #"Given the following objective: 'Compare three data visualization libraries and "
    #"evaluate them in terms of ease of use, aesthetics, customization, interactivity, "
    #"evaluate them in terms of statistical support, integration with Jupyter or Dash, and select the best one for general use', "
#    "choose one visualization library and explain why it is the best choice."
    #"Also, provide a markdown table comparing the three libraries based on the criteria mentioned."
#)

# Correr el agente FSM
agent_fsm = AutonomousAgentFSM(
    goal=prompt,
    llm=llm,
    retriever=retriever
    #, texts=texts
)
output = agent_fsm.run()
print("\n=== Resultado final ===\n", output)

🔧 Ejecutando subtarea: Search information about Compare the three visualization libraries considering:
- Ease of use
- Aesthetics
- Personalization
- Interactivity
- Statistical support
- Jupyter/Dash integration

Make a final one recommendation for general use.
🔧 Ejecutando subtarea: Compare the data of three visualization libraries
🔧 Ejecutando subtarea: recommend the best visualization library for python

=== Resultado final ===
 ✅ Evaluación de resultados:
- Statistical support
- Matplotlib — Visualization with Python Skip to main content Ctrl + K
- Matplotlib
